# 03 · Architecture: GRU vs LSTM, depth, bidirectionality
Point 3 of `PLAN.md` (owner: Mykola):
1. Cross **cell type** (GRU vs LSTM) × **depth** (1/2/3 stacked layers) × **direction** (uni vs bi) — 12 runs.
2. Report CV/es curves and parameter counts for every combination.
3. A dedicated **gradient-flow** check across depth: does a 3-layer stack still train cleanly, or does the classic vanishing/exploding-gradient problem show up, and does LSTM's gating resist it better than GRU's?
4. Feed the winning architecture into point 4 (optimizers/schedulers/hyperparameters, owner: Mykola too).

**Why BPE 16k / last-piece labels:** point 5 (`05_tokenization.ipynb`) already crossed tokenizer × alignment × direction with a 1-layer GRU and recommended BPE 16k with the label on the last piece as the default for points 3 and 4, confirmed on the LB (+0.078 over the word-level baseline). Re-deriving that here would just burn compute, so this notebook takes it as given and holds it fixed while varying architecture.

**Everything else is held at the point-2/5 baseline** (hidden 256, embeddings 128 random, dropout 0.3, AdamW lr 2e-3, weight decay 0.01, batch 256, up to 6 epochs, early stopping on `es_f1` patience 2, seed 42) so the grid isolates the architecture effect and stays comparable with the rest of the team's results log.

**How to run:** on Colab with a T4. The grid is 12 runs; a BiGRU/BiLSTM epoch on BPE 16k took ~4–5 min in point 5, and depth/cell-type roughly scale that up, so **expect on the order of 5–7 hours for the full grid** — deliberately long, matching the unstable-power note (leave it running, it resumes from `ARTIFACTS/{run_name}/report.json` if interrupted). Trim `GRID` in section 2 to shorten it. Set `QUICK_RUN=1` to check the code on a tiny subset in a few minutes, on CPU.

In [ ]:
NOTEBOOK = "03-architecture"  # W&B group of this notebook's runs

## 0. Shared code
Section 0 is **identical in every model notebook** (copied verbatim from the master copy in `02_validation_baseline.ipynb`), so this notebook uses the same splits, metric, tokenizer, training loop and output format as points 2 and 5. Code specific to this notebook goes in its own sections below.

### 0.0 Setup
- **Colab:** upload `secrets.env` (`KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`) to `/content/` first. The cell loads the keys (it prints only their names), installs missing packages, and downloads the data with the Kaggle CLI. If Google Drive is mounted at `/content/drive`, run outputs go to `MyDrive/dl-a3-artifacts` so they survive the VM reset — make that folder a shortcut to the team's shared folder, so everyone writes to the same results log.
- **Locally:** the data is read from `data/raw/`, outputs go to `artifacts/` (git-ignored).
- `QUICK_RUN=1` (environment variable) trains on a small subset for one epoch, to check the code in a few minutes, on CPU.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

COMPETITION = "kse-ua-location-extraction-2026"
DATA_FILES = ["uk_geo_dataset.csv", "labeling_sample.csv", "test.csv"]
IN_COLAB = Path("/content").exists()
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"


def load_secrets(path: Path = Path("/content/secrets.env")) -> None:
    assert path.exists(), f"Upload secrets.env to {path.parent} first"
    for line in path.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()
    print("keys loaded:", [k for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "WANDB_API_KEY") if os.environ.get(k)])


def download_data(raw: Path) -> None:
    raw.mkdir(parents=True, exist_ok=True)
    for name in DATA_FILES:
        if (raw / name).exists():
            continue
        subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION, "-f", name, "-p", str(raw)], check=True)
        archive = raw / f"{name}.zip"
        if archive.exists():  # large files arrive zipped
            shutil.unpack_archive(archive, raw)
            archive.unlink()


def install_missing(packages: dict[str, str]) -> None:
    """{import name: pip name}; installs only what is missing."""
    missing = [pip for module, pip in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


install_missing({"lightning": "lightning", "tokenizers": "tokenizers", "wandb": "wandb", "kaggle": "kaggle"})
if IN_COLAB:
    load_secrets()
    RAW = Path("/content/data/raw")
    download_data(RAW)
    drive = Path("/content/drive/MyDrive")
    ARTIFACTS = (drive / "dl-a3-artifacts") if drive.exists() else Path("/content/artifacts")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RAW, ARTIFACTS = ROOT / "data/raw", ROOT / "artifacts"
if QUICK_RUN:
    ARTIFACTS = ARTIFACTS / "quick_run"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"data: {RAW} | outputs: {ARTIFACTS} | quick run: {QUICK_RUN}")

### 0.1 Tokenizer, span ↔ BIO, competition metric
Every training notebook uses the same code. Each token keeps its character offsets, so predicted tags can be turned back into **exact substrings** of the text, which is what the metric compares. Punctuation becomes its own token (`Київ,` → `Київ` `,`), while inner hyphens and apostrophes stay inside the word (`Івано-Франківськ`, `Слов'янськ`).

In [ ]:
import ast
import re
import string
from collections import Counter
from dataclasses import dataclass
from typing import List, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

TOKEN_RE = re.compile(r"\w+(?:[-'’ʼ`]\w+)*|[^\w\s]")


@dataclass(frozen=True)
class Token:
    text: str
    start: int
    end: int  # exclusive


def word_tokenize(text: str) -> list[Token]:
    return [Token(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


def parse_markers(value) -> list[tuple[int, int]]:
    """`loc_markers` as stored in the CSV ("[(0, 4), (10, 15)]") -> [(start, end), ...], end exclusive."""
    if not isinstance(value, str):
        return []
    return [(int(a), int(b)) for a, b in ast.literal_eval(value.strip() or "[]")]


def spans_to_bio(tokens: Sequence[Token], spans) -> list[str]:
    """Every token that overlaps a span is labelled: the first one B-LOC, the rest I-LOC."""
    labels = ["O"] * len(tokens)
    for left, right in sorted(spans):
        first = True
        for i, tok in enumerate(tokens):
            if tok.start < right and tok.end > left:
                labels[i] = "B-LOC" if first else "I-LOC"
                first = False
    return labels


def bio_to_spans(tokens: Sequence[Token], tags: Sequence[str]) -> list[tuple[int, int]]:
    """Merge BIO tags into character spans; an I-LOC without an open entity starts a new one."""
    spans, current = [], None
    for tok, tag in zip(tokens, tags):
        if tag == "B-LOC" or (tag == "I-LOC" and current is None):
            if current is not None:
                spans.append(tuple(current))
            current = [tok.start, tok.end]
        elif tag == "I-LOC":
            current[1] = tok.end
        else:
            if current is not None:
                spans.append(tuple(current))
            current = None
    if current is not None:
        spans.append(tuple(current))
    return spans


def spans_to_strings(text: str, spans) -> list[str]:
    """Cut location strings out of the original text (deduplicated, order kept)."""
    return list(dict.fromkeys(s for s in (text[a:b].strip() for a, b in spans) if s))

`hard_processing` and `comp_metric` below are copied **verbatim** from the competition page. Do not "fix" them.

In [ ]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p,  = 0.0, 0.0, 0.0, 0.0

    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])

        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)

    if tp + fp == 0:
        if p == 0:
            precision = 1.0
        else:
            precision = 0.0
    else:
        precision = tp/(tp + fp)

    if tp + fn == 0:
        if p == 0:
            recall = 1.0
        else:
            recall = 0.0
    else:
        recall = tp/(tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

Complementary metrics (notebook 01, section 8). They count exactly like `comp_metric`, plus:

In [ ]:
def precision_recall(y_true, y_pred) -> tuple[float, float]:
    tp = fp = fn = 0
    for t_s, p_s in zip(y_true, y_pred):
        t, p = {hard_processing(s) for s in t_s}, {hard_processing(s) for s in p_s}
        tp, fp, fn = tp + len(t & p), fp + len(p - t), fn + len(t - p)
    precision = tp / (tp + fp) if tp + fp else float(tp + fn == 0)
    recall = tp / (tp + fn) if tp + fn else 1.0
    return precision, recall


def partial_match_f1(y_true, y_pred) -> float:
    """Boundary-tolerant F1: a location counts if it shares a normalized word with one on the other side."""
    words = lambda s: set(hard_processing(s).split())
    hit_p = n_p = hit_t = n_t = 0
    for t_s, p_s in zip(y_true, y_pred):
        t_sets = [words(s) for s in {hard_processing(x) for x in t_s}]
        p_sets = [words(s) for s in {hard_processing(x) for x in p_s}]
        n_p, n_t = n_p + len(p_sets), n_t + len(t_sets)
        hit_p += sum(any(p & t for t in t_sets) for p in p_sets)
        hit_t += sum(any(t & p for p in p_sets) for t in t_sets)
    precision, recall = (hit_p / n_p if n_p else 1.0), (hit_t / n_t if n_t else 1.0)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def metric_report(y_true, y_pred) -> dict:
    precision, recall = precision_recall(y_true, y_pred)
    no_loc = [len(p) > 0 for t, p in zip(y_true, y_pred) if not t]
    with_loc = [(t, p) for t, p in zip(y_true, y_pred) if t]
    return {
        "comp_f1": comp_metric(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "partial_f1": partial_match_f1(y_true, y_pred),
        "f1_texts_with_loc": comp_metric([t for t, _ in with_loc], [p for _, p in with_loc]),
        "fp_rate_texts_without_loc": float(np.mean(no_loc)) if no_loc else 0.0,
    }


def bootstrap_ci(y_true, y_pred, n_boot: int = 2000, seed: int = 0, alpha: float = 0.05) -> tuple[float, float]:
    """95% interval of comp_metric when the texts are resampled with replacement (CV has only 26 texts)."""
    rng, n = np.random.default_rng(seed), len(y_true)
    scores = [comp_metric([y_true[i] for i in idx], [y_pred[i] for i in idx]) for idx in rng.integers(0, n, (n_boot, n))]
    low, high = np.quantile(scores, [alpha / 2, 1 - alpha / 2])
    return float(low), float(high)

### 0.2 Data: corpora and encoded datasets
- A `Corpus` holds the texts, word tokens and BIO tags (label ids) of one split, and the gold strings. For evaluation splits it also keeps the tokens with their offsets, which decoding needs (train skips them to save memory).
- A **vocabulary** maps words to ids. Any vocabulary works if it has `encode_batch(word_lists) -> [(ids, first_pos, last_pos), ...]`, where `first_pos[w]` and `last_pos[w]` are the positions of word `w`'s first and last id. For word-level they are equal. For BPE a word can have several pieces.
- `EncodedDataset` puts each word's label on its first or last piece (`alignment`). Every other position gets −100, which the loss ignores. Training sequences are cut at `max_len` ids. Evaluation sequences are **never cut**, because test posts can be long (notebook 01).

In [ ]:
import json
import logging
import time
import warnings
from collections import Counter

import lightning as L
import torch
import torch.nn as nn
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.utilities import grad_norm
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", module="lightning")
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["O", "B-LOC", "I-LOC"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
PAD_TOKEN, UNK_TOKEN, PAD_ID, UNK_ID, IGNORE_INDEX = "<pad>", "<unk>", 0, 1, -100
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


class Corpus:
    def __init__(self, texts, spans=None, keep_tokens: bool = True):
        texts = [str(t) for t in texts]
        tokens = [word_tokenize(t) for t in texts]
        self.words = [[t.text for t in toks] for toks in tokens]
        self.tags = None if spans is None else [
            np.array([LABEL2ID[tag] for tag in spans_to_bio(toks, s)], dtype=np.int64) for toks, s in zip(tokens, spans)]
        self.gold = None if spans is None else [[t[a:b] for a, b in s] for t, s in zip(texts, spans)]
        self.texts, self.tokens = texts, (tokens if keep_tokens else None)

    def __len__(self) -> int:
        return len(self.words)


class WordVocab:
    """Word -> id, built on training words only; the `max_size` most frequent words seen `min_freq`+ times; the rest
    -> <unk>. `lowercase=False` keeps case (`Київ` and `київ` become different words)."""

    def __init__(self, word_lists, max_size: int = 50_000, min_freq: int = 2, lowercase: bool = True):
        self.norm = str.lower if lowercase else str
        counts = Counter(self.norm(w) for words in word_lists for w in words)
        self.itos = [PAD_TOKEN, UNK_TOKEN] + [w for w, c in counts.most_common(max_size - 2) if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self) -> int:
        return len(self.itos)

    def encode_batch(self, word_lists):
        for words in word_lists:
            ids = [self.stoi.get(self.norm(w), UNK_ID) for w in words]
            positions = list(range(len(words)))
            yield ids, positions, positions


class EncodedDataset(Dataset):
    def __init__(self, corpus: Corpus, vocab, alignment: str = "first", max_len: int | None = None):
        assert alignment in ("first", "last")
        self.items = []
        for i, (ids, first, last) in enumerate(vocab.encode_batch(corpus.words)):
            ids, pos = np.asarray(ids, dtype=np.int64), np.asarray(first if alignment == "first" else last, dtype=np.int64)
            labels = np.full(len(ids), IGNORE_INDEX, dtype=np.int64)
            if corpus.tags is not None:
                labels[pos] = corpus.tags[i]
            if max_len is not None and len(ids) > max_len:
                ids, labels, pos = ids[:max_len], labels[:max_len], pos[pos < max_len]
            self.items.append((ids, labels, pos))

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        return (idx, *self.items[idx])


def collate(batch) -> dict:
    """Right-pad ids with PAD_ID and labels with −100; `lengths` feed pack_padded_sequence."""
    lengths = torch.tensor([max(len(ids), 1) for _, ids, _, _ in batch])
    input_ids = torch.full((len(batch), int(lengths.max())), PAD_ID, dtype=torch.long)
    labels = torch.full_like(input_ids, IGNORE_INDEX)
    for row, (_, ids, lab, _) in enumerate(batch):
        input_ids[row, :len(ids)] = torch.from_numpy(ids)
        labels[row, :len(lab)] = torch.from_numpy(lab)
    return {"idx": [b[0] for b in batch], "input_ids": input_ids, "labels": labels, "lengths": lengths,
            "positions": [b[3] for b in batch]}

**Splits** (point 2.1, notebook 02). They are fixed by the data, so they need no seed:
- **train** = `uk_geo_dataset` with `is_valid==0` (1M sentences);
- **es** = `is_valid==1` (10k sentences, train domain): early stopping and checkpoint choice;
- **CV** = `labeling_sample` (26 test-like texts): the score we report and compare with the LB;
- **test** = 477 posts for the submission.

`QUICK_RUN` samples 20k train and 2k es sentences with a fixed seed.

In [ ]:
def load_corpora() -> tuple[dict[str, Corpus], pd.DataFrame]:
    uk = pd.read_csv(RAW / "uk_geo_dataset.csv", usecols=["text", "loc_markers", "is_valid"])
    ls = pd.read_csv(RAW / "labeling_sample.csv")
    test = pd.read_csv(RAW / "test.csv")
    train_df, es_df = uk[uk.is_valid == 0], uk[uk.is_valid == 1]
    if QUICK_RUN:
        train_df, es_df = train_df.sample(20_000, random_state=0), es_df.sample(2_000, random_state=0)
    corpora = {
        "train": Corpus(train_df.text, train_df.loc_markers.map(parse_markers), keep_tokens=False),
        "es": Corpus(es_df.text, es_df.loc_markers.map(parse_markers)),
        "cv": Corpus(ls.text, ls.loc_markers.map(parse_markers)),
        "test": Corpus(test.text),
    }
    return corpora, test

### 0.3 Model
Embedding → GRU / LSTM / RNN (optionally bidirectional, several layers) → a linear layer on every position. `pack_padded_sequence` makes the RNN stop at each text's real length, so padding never enters the hidden state, not even in the backward direction. This class already supports everything point 3 needs (`rnn_type`, `num_layers`, `bidirectional`) — nothing to change here.

In [ ]:
RNN_TYPES = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}


class RNNTagger(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 256, rnn_type: str = "gru",
                 num_layers: int = 1, bidirectional: bool = True, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.rnn = RNN_TYPES[rnn_type](embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                                       bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Linear(hidden_dim * (2 if bidirectional else 1), len(LABELS))

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        x = self.dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=input_ids.size(1))
        return self.classifier(self.dropout(out))  # [batch, time, labels]

### 0.4 Training and decoding
- Loss: token-level cross-entropy, ignoring −100.
- After every epoch, both validation sets are decoded to strings and scored with the **competition metric**:
  - `es_f1` (train domain) selects the checkpoint and drives early stopping;
  - `cv_f1` (test-like) is only reported. Choosing epochs on 26 CV texts would overfit them.
- Gradient norms per parameter (before clipping) are logged every `log_every_n_steps` steps: the gradient-flow check this notebook relies on in section 6.
- Decoding: word-level probabilities → argmax tag per word → BIO → character spans → strings cut from the text.

In [ ]:
def word_probs(logits: torch.Tensor, positions) -> list[np.ndarray]:
    """[batch, time, labels] logits -> per text a [n_words, labels] probability array, read at the aligned piece."""
    probs = logits.float().softmax(-1).detach().cpu().numpy()
    return [probs[row, pos] for row, pos in enumerate(positions)]


def decode(corpus: Corpus, probs: list[np.ndarray]) -> list[list[str]]:
    tags = [[ID2LABEL[int(i)] for i in p.argmax(-1)] for p in probs]
    return [spans_to_strings(text, bio_to_spans(toks, t)) for text, toks, t in zip(corpus.texts, corpus.tokens, tags)]


class TaggerModule(L.LightningModule):
    def __init__(self, model: nn.Module, lr: float, weight_decay: float, val_corpora: dict[str, Corpus]):
        super().__init__()
        self.model, self.lr, self.weight_decay = model, lr, weight_decay
        self.val_names, self.val_corpora = list(val_corpora), list(val_corpora.values())
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def _loss(self, logits, labels):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

    def training_step(self, batch, batch_idx):
        loss = self._loss(self.model(batch["input_ids"], batch["lengths"]), batch["labels"])
        self.log("train_loss", loss, on_step=True, on_epoch=True, batch_size=len(batch["idx"]))
        return loss

    def on_validation_epoch_start(self):
        self.val_probs = [{} for _ in self.val_names]

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        logits = self.model(batch["input_ids"], batch["lengths"])
        name = self.val_names[dataloader_idx]
        self.log(f"{name}_loss", self._loss(logits, batch["labels"]), batch_size=len(batch["idx"]),
                 add_dataloader_idx=False)
        self.val_probs[dataloader_idx].update(zip(batch["idx"], word_probs(logits, batch["positions"])))

    def on_validation_epoch_end(self):
        for name, corpus, buf in zip(self.val_names, self.val_corpora, self.val_probs):
            pred = decode(corpus, [buf[i] for i in range(len(corpus))])
            precision, recall = precision_recall(corpus.gold, pred)
            self.log_dict({f"{name}_f1": comp_metric(corpus.gold, pred), f"{name}_precision": precision,
                           f"{name}_recall": recall})

    def on_before_optimizer_step(self, optimizer):  # before clipping; Lightning writes it every log_every_n_steps
        self.log_dict(grad_norm(self.model, norm_type=2))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)


@torch.no_grad()
def predict(model: nn.Module, dataset: EncodedDataset, batch_size: int = 256) -> list[np.ndarray]:
    model.eval().to(DEVICE)
    out = [None] * len(dataset)
    for batch in DataLoader(dataset, batch_size=batch_size, collate_fn=collate):
        logits = model(batch["input_ids"].to(DEVICE), batch["lengths"])
        for i, p in zip(batch["idx"], word_probs(logits, batch["positions"])):
            out[i] = p
    return out

### 0.5 One experiment
`run_experiment(cfg, vocab, corpora, test_ids)`: train → reload the best checkpoint (by `es_f1`) → predict CV and test → CV report → save outputs. If the run's `report.json` already exists it is loaded instead, so a crashed or disconnected session (relevant with unstable power) can resume. Outputs in `ARTIFACTS/{run_name}/`: `best.ckpt`, `config.json`, `logs/metrics.csv` (curves), `cv_probs.pkl` / `test_probs.pkl`, `submission.csv`, `report.json`.

In [ ]:
import pickle

WANDB_PROJECT = "kse-ua-location-extraction"


def save_probs(probs, path: Path) -> None:
    with open(path, "wb") as f:
        pickle.dump([p.astype(np.float16) for p in probs], f)


def load_probs(run: str, split: str) -> list[np.ndarray]:
    """Word-level probabilities a run saved for `split` ("cv" or "test")."""
    with open(ARTIFACTS / run / f"{split}_probs.pkl", "rb") as f:
        return [p.astype(np.float32) for p in pickle.load(f)]


def write_submission(text_ids, locations, path: Path) -> None:
    pd.DataFrame({"text_id": text_ids,
                  "locations": [json.dumps(list(x), ensure_ascii=False) for x in locations]}).to_csv(path, index=False)


def run_experiment(cfg: dict, vocab, corpora: dict[str, Corpus], test_ids, force: bool = False) -> dict:
    out = ARTIFACTS / cfg["run_name"]
    if (out / "report.json").exists() and not force:
        return json.loads((out / "report.json").read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    L.seed_everything(cfg["seed"], workers=True)
    start = time.time()

    encode = lambda name, max_len=None: EncodedDataset(corpora[name], vocab, cfg["alignment"], max_len)
    train_ds, es_ds, cv_ds, test_ds = encode("train", cfg["max_len"]), encode("es"), encode("cv"), encode("test")
    workers = 2 if DEVICE == "cuda" else 0
    loader = lambda ds, shuffle=False: DataLoader(ds, batch_size=cfg["batch_size"], shuffle=shuffle,
                                                  collate_fn=collate, num_workers=workers)

    model = RNNTagger(len(vocab), **cfg["model"])
    module = TaggerModule(model, cfg["lr"], cfg["weight_decay"], {"es": corpora["es"], "cv": corpora["cv"]})
    loggers = [CSVLogger(ARTIFACTS, name=cfg["run_name"], version="logs")]
    if os.environ.get("WANDB_API_KEY") and not QUICK_RUN:
        loggers.append(WandbLogger(project=WANDB_PROJECT, name=cfg["run_name"], group=NOTEBOOK, config=cfg,
                                   save_dir=str(out)))
    checkpoint = ModelCheckpoint(dirpath=out, filename="best", monitor="es_f1", mode="max", save_top_k=1)
    trainer = L.Trainer(max_epochs=cfg["max_epochs"], accelerator="gpu" if DEVICE == "cuda" else "cpu", devices=1,
                        gradient_clip_val=1.0, log_every_n_steps=10 if QUICK_RUN else 100, num_sanity_val_steps=0,
                        logger=loggers,
                        callbacks=[checkpoint, EarlyStopping(monitor="es_f1", mode="max", patience=cfg["patience"])],
                        enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(module, loader(train_ds, shuffle=True), [loader(es_ds), loader(cv_ds)])

    best = torch.load(checkpoint.best_model_path, map_location="cpu", weights_only=False)
    model.load_state_dict({k.removeprefix("model."): v for k, v in best["state_dict"].items()})
    cv_probs, test_probs = predict(model, cv_ds), predict(model, test_ds)
    save_probs(cv_probs, out / "cv_probs.pkl")
    save_probs(test_probs, out / "test_probs.pkl")
    write_submission(test_ids, decode(corpora["test"], test_probs), out / "submission.csv")

    report = {"run_name": cfg["run_name"], **metric_report(corpora["cv"].gold, decode(corpora["cv"], cv_probs)),
              "es_f1": float(checkpoint.best_model_score), "best_epoch": int(best["epoch"]),
              "epochs_run": trainer.current_epoch, "vocab_size": len(vocab),
              "params": sum(p.numel() for p in model.parameters()), "minutes": (time.time() - start) / 60}
    (out / "report.json").write_text(json.dumps(report, indent=2))
    if len(loggers) > 1:
        loggers[1].experiment.summary.update({f"final_cv_{k}": v for k, v in report.items() if k != "run_name"})
        loggers[1].experiment.finish()
    return report

### 0.6 Results log and submissions
`ARTIFACTS/experiments.csv` holds one row per run: `run_name, owner, point, description, cv_f1, cv_p, cv_r, es_f1, lb_public, notes`. Failed runs stay in. `log_result` replaces the run's row and keeps an LB score that is already there. `submit` sends a run's `submission.csv` to Kaggle, waits for the public score, and writes it into the log. The team has 10 submissions a day (Ivan 3, Mykola 4, Taras 3) — this notebook never submits automatically; `SUBMIT_RUN` has to be set by hand after reviewing the results.

In [ ]:
import io

RESULTS_LOG = ARTIFACTS / "experiments.csv"
LOG_COLUMNS = ["run_name", "owner", "point", "description", "cv_f1", "cv_p", "cv_r", "es_f1", "lb_public", "notes"]


def read_log() -> pd.DataFrame:
    return pd.read_csv(RESULTS_LOG) if RESULTS_LOG.exists() else pd.DataFrame(columns=LOG_COLUMNS)


def log_result(cfg: dict, report: dict, lb_public: float | None = None, notes: str = "") -> pd.DataFrame:
    log = read_log()
    old = log[log.run_name == cfg["run_name"]]
    if lb_public is None and len(old):
        lb_public = old.lb_public.iloc[0]
    row = {"run_name": cfg["run_name"], "owner": cfg["owner"], "point": cfg["point"],
           "description": cfg.get("description", ""), "cv_f1": report["comp_f1"], "cv_p": report["precision"],
           "cv_r": report["recall"], "es_f1": report["es_f1"], "lb_public": lb_public, "notes": notes}
    others = log[log.run_name != cfg["run_name"]]
    log = pd.DataFrame([*others.to_dict("records"), row], columns=LOG_COLUMNS)  # no concat: no all-NA dtype warning
    if not QUICK_RUN:
        log.to_csv(RESULTS_LOG, index=False)
    return log


def kaggle_public_score(message: str, timeout_s: int = 300) -> float | None:
    """Poll our Kaggle submissions until the one with this message has a public score."""
    for _ in range(timeout_s // 15):
        time.sleep(15)
        lines = subprocess.run(["kaggle", "competitions", "submissions", "-c", COMPETITION, "--csv"],
                               capture_output=True, text=True, check=True).stdout.splitlines()
        header = next(i for i, line in enumerate(lines) if "publicScore" in line)  # skip CLI warnings
        subs = pd.read_csv(io.StringIO("\n".join(lines[header:])))
        mine = subs[subs.description.astype(str) == message]
        if len(mine) and pd.notna(mine.publicScore.iloc[0]):  # newest first
            return float(mine.publicScore.iloc[0])
        if len(mine) and "error" in str(mine.status.iloc[0]).lower():
            raise RuntimeError(f"Kaggle rejected the submission: {mine.status.iloc[0]}")
    print("no public score yet; check Kaggle and call log_result(cfg, report, lb_public=...) by hand")
    return None


def submit(cfg: dict, report: dict, notes: str = "") -> float | None:
    path = ARTIFACTS / cfg["run_name"] / "submission.csv"
    message = f"{cfg['run_name']} | CV {report['comp_f1']:.3f}"
    subprocess.run(["kaggle", "competitions", "submit", "-c", COMPETITION, "-f", str(path), "-m", message], check=True)
    lb = kaggle_public_score(message)
    log_result(cfg, report, lb_public=lb, notes=notes)
    print(f"{cfg['run_name']}: public LB = {lb}")
    return lb

### 0.7 BPE 16k vocabulary (copied from `05_tokenization.ipynb`, point 5, owner Ivan)
Point 5's conclusion: *"Use BPE 16k with last-piece labels as the default tokenizer for architecture and optimizer experiments (points 3 and 4)."* This notebook follows that recommendation, so `BpeVocab` is copied here unchanged rather than re-deriving it.

In [ ]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers


class BpeVocab:
    def __init__(self, word_lists, vocab_size: int, min_freq: int = 2):
        tok = Tokenizer(models.BPE(unk_token=UNK_TOKEN, continuing_subword_prefix="##"))
        tok.normalizer = normalizers.Lowercase()
        tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()  # our words contain no spaces
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_freq, show_progress=False,
                                      special_tokens=[PAD_TOKEN, UNK_TOKEN], continuing_subword_prefix="##")
        tok.train_from_iterator((" ".join(words) for words in word_lists), trainer)
        assert tok.token_to_id(PAD_TOKEN) == PAD_ID and tok.token_to_id(UNK_TOKEN) == UNK_ID
        self.tokenizer = tok

    def __len__(self) -> int:
        return self.tokenizer.get_vocab_size()

    def pieces(self, word: str) -> list[str]:
        return self.tokenizer.encode([word], is_pretokenized=True, add_special_tokens=False).tokens

    def encode_batch(self, word_lists, chunk_size: int = 10_000):
        for start in range(0, len(word_lists), chunk_size):
            chunk = word_lists[start:start + chunk_size]
            encodings = self.tokenizer.encode_batch([w or ["."] for w in chunk], is_pretokenized=True,
                                                    add_special_tokens=False)
            for words, enc in zip(chunk, encodings):
                if not words:
                    yield [], [], []
                    continue
                first, last = [None] * len(words), [None] * len(words)
                for pos, w in enumerate(enc.word_ids):
                    if first[w] is None:
                        first[w] = pos
                    last[w] = pos
                assert None not in first, "a word produced no pieces"
                yield enc.ids, first, last

## 1. Data
The shared splits from section 0, plus the BPE 16k vocabulary (section 0.7), built on the full training set. Training the BPE tokenizer on 1M sentences takes a couple of minutes.

In [ ]:
corpora, test = load_corpora()
print({name: len(c) for name, c in corpora.items()})

In [ ]:
start = time.time()
bpe_vocab = BpeVocab(corpora["train"].words, vocab_size=16_000)
print(f"bpe16k vocab: {len(bpe_vocab):,} pieces, built in {time.time() - start:.0f}s")

## 2. Architecture grid
At every word $t$, GRU and LSTM both replace the vanilla RNN's single $\tanh$ update with learned gates that decide how much of the previous state to keep vs overwrite, which is what is supposed to let gradients flow through many steps without vanishing as fast as a plain RNN's does (point 2's baseline already showed a 1-layer RNN's *parameter* gradient norms were stable, but that check cannot see vanishing *through time* inside BPTT — section 6 below looks at vanishing **through stacked layers** instead, which per-parameter norms *can* show).

- **GRU:** one update gate $z_t$, one reset gate $r_t$; fewer parameters than LSTM for the same hidden size (no separate cell state).
- **LSTM:** input/forget/output gates plus a separate cell state $c_t$; the forget gate is specifically designed to let gradients pass through the cell state almost unchanged when it is close to 1.
- **Depth (`num_layers`):** stacking layers lets the model build higher-level features of the sequence, at the cost of a longer path for gradients to travel back to the embedding layer — exactly what section 6 checks.
- **Bidirectional:** the backward pass lets a word's tag depend on what comes *after* it too (`Фастів - 11`: the list format only makes sense once you've seen the number). Point 5 found this helps `es_f1` much more than `cv_f1`, which this grid re-checks at every depth.

In [ ]:
BASE = {"seed": 42, "max_len": 256, "batch_size": 256, "lr": 2e-3, "weight_decay": 0.01,
        "max_epochs": 1 if QUICK_RUN else 6, "patience": 2,
        "tokenizer": "bpe16k", "alignment": "last",  # point 5's recommendation for points 3-4
        "model": {"embed_dim": 128, "hidden_dim": 256, "dropout": 0.3}}

GRID = [  # (rnn_type, num_layers, bidirectional)
    ("gru", 1, False), ("gru", 1, True),
    ("gru", 2, False), ("gru", 2, True),
    ("gru", 3, False), ("gru", 3, True),
    ("lstm", 1, False), ("lstm", 1, True),
    ("lstm", 2, False), ("lstm", 2, True),
    ("lstm", 3, False), ("lstm", 3, True),
]
if QUICK_RUN:
    GRID = GRID[:2]  # just check the code runs


def make_config(rnn_type: str, num_layers: int, bidirectional: bool) -> dict:
    arch = f"{'bi' if bidirectional else 'uni'}{rnn_type}{num_layers}"
    direction = "bidirectional" if bidirectional else "unidirectional"
    cfg = {**BASE, "owner": "mykola", "point": 3, "run_name": f"mykola-{arch}-bpe16k-last",
           "description": f"{num_layers}-layer {direction} {rnn_type.upper()}, BPE 16k, "
                          f"label on last piece, AdamW"}
    cfg["model"] = {**BASE["model"], "rnn_type": rnn_type, "num_layers": num_layers,
                   "bidirectional": bidirectional}
    return cfg

In [ ]:
reports = []
for rnn_type, num_layers, bidirectional in GRID:
    cfg = make_config(rnn_type, num_layers, bidirectional)
    report = run_experiment(cfg, bpe_vocab, corpora, test.text_id)
    reports.append({"rnn_type": rnn_type, "num_layers": num_layers,
                    "direction": "bi" if bidirectional else "uni", **report})
    print(f"{cfg['run_name']:<22} cv_f1={report['comp_f1']:.3f}  es_f1={report['es_f1']:.3f}  "
          f"params={report['params']:,}  ({report['minutes']:.1f} min)")

## 3. Results
Saved to `results_03_architecture.csv` — notebook 04 reads this to pick up the winning architecture, so keep the column names (`rnn_type`, `num_layers`, `direction`, `comp_f1`, ...) as they are.

In [ ]:
results = pd.DataFrame(reports).set_index("run_name")
results.to_csv(ARTIFACTS / "results_03_architecture.csv")
cols = ["rnn_type", "num_layers", "direction", "comp_f1", "precision", "recall", "partial_f1", "es_f1",
        "best_epoch", "params", "minutes"]
results[cols].round(3)

In [ ]:
results.pivot_table(index=["rnn_type", "num_layers"], columns="direction",
                    values=["comp_f1", "es_f1", "params"]).round(3)

Reference points already in the team's log, for scale: point 2's vanilla-RNN baseline (word-level, 1 layer, unidirectional) scored **CV 0.488 / es 0.802**; point 5's best (BiGRU, 1 layer, BPE 16k, last-piece) scored **CV 0.597 / es 0.850 / LB 0.537** and is the `bigru1` row in this grid repeated exactly (same config), so it should reproduce that score up to training-run noise.

## 4. Curves
Solid = bidirectional, dashed = unidirectional; blue = GRU, orange = LSTM; line width grows with depth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
COLORS = {"gru": "tab:blue", "lstm": "tab:orange"}
for run in results.index:
    row = results.loc[run]
    log = pd.read_csv(ARTIFACTS / run / "logs" / "metrics.csv")
    epochs = log.dropna(subset=["es_f1"]).groupby("epoch")[["es_f1", "cv_f1"]].last()
    style = "-" if row.direction == "bi" else "--"
    lw = 0.8 + 0.7 * row.num_layers
    axes[0].plot(epochs.index + 1, epochs.es_f1, style, color=COLORS[row.rnn_type], lw=lw, marker="o", ms=3)
    axes[1].plot(epochs.index + 1, epochs.cv_f1, style, color=COLORS[row.rnn_type], lw=lw, marker="o", ms=3,
                 label=run)
axes[0].set(title="es F1 (train domain, selects the checkpoint)", xlabel="epoch")
axes[1].set(title="CV F1 (test-like, reported)", xlabel="epoch")
axes[1].legend(frameon=False, fontsize=6, ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for run in results.index:
    row = results.loc[run]
    log = pd.read_csv(ARTIFACTS / run / "logs" / "metrics.csv")
    per_epoch = log.groupby("epoch").last()
    style = "-" if row.direction == "bi" else "--"
    lw = 0.8 + 0.7 * row.num_layers
    axes[0].plot(per_epoch.index + 1, per_epoch["train_loss_epoch"], style, color=COLORS[row.rnn_type], lw=lw)
    axes[1].plot(per_epoch.index + 1, per_epoch["cv_loss"], style, color=COLORS[row.rnn_type], lw=lw, label=run)
axes[0].set(title="train loss", xlabel="epoch")
axes[1].set(title="CV loss (test-like)", xlabel="epoch")
axes[1].legend(frameon=False, fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Parameter count vs score
LSTM has ~4× the gate parameters of a vanilla RNN cell per layer (GRU: 3×), so depth and cell type both grow the model — worth checking whether extra parameters actually buy CV F1, or just overfit the train domain (raise `es_f1` without raising `cv_f1`, the same pattern point 2 found from *more training*, now checked for *more capacity*).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for rnn_type, marker in [("gru", "o"), ("lstm", "s")]:
    for direction, fc in [("uni", "none"), ("bi", COLORS[rnn_type])]:
        sub = results[(results.rnn_type == rnn_type) & (results.direction == direction)]
        ax.scatter(sub.params, sub.comp_f1, marker=marker, s=60, facecolors=fc,
                   edgecolors=COLORS[rnn_type], label=f"{rnn_type} {direction}")
        for run, row in sub.iterrows():
            ax.annotate(f"L{row.num_layers}", (row.params, row.comp_f1), fontsize=7,
                       xytext=(4, 2), textcoords="offset points")
ax.set(xlabel="parameters", ylabel="CV F1 (comp_metric)", title="capacity vs CV score")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 6. Gradient flow across depth
`on_before_optimizer_step` (section 0.4) logs the L2 norm of every parameter's gradient, before clipping, every `log_every_n_steps`. For a stacked RNN the recurrent weight matrix of each layer (`rnn.weight_hh_l0`, `weight_hh_l1`, ...; `_reverse` for the backward direction of a bidirectional model) is exactly where vanishing/exploding gradients across **depth** would show up: if the earliest layer's gradient norm is orders of magnitude smaller than the last layer's throughout training, gradients are struggling to reach the layers closest to the embeddings.

This plots the **deepest unidirectional model actually in the grid** for each cell type (GRU and LSTM) — the pair most likely to show the effect, since depth is the whole point of this check and a backward pass would re-inject gradients from the other direction, masking it. Picked dynamically from `results` rather than hardcoded run names, so this cell still works if `GRID` above was trimmed (e.g. under `QUICK_RUN`, or because the grid was shortened to save time on unstable power).

In [ ]:
import re

LAYER_COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]


def plot_gradient_flow(run: str, ax, title: str) -> None:
    metrics = pd.read_csv(ARTIFACTS / run / "logs" / "metrics.csv")
    for col in [c for c in metrics.columns if c.startswith("grad_2.0_norm/") and "weight_hh" in c]:
        name = col.split("/", 1)[1]
        layer = int(re.search(r"_l(\d+)", name).group(1))
        style = "--" if "reverse" in name else "-"
        series = metrics[["step", col]].dropna()
        ax.plot(series.step, series[col], style, color=LAYER_COLORS[layer % len(LAYER_COLORS)], lw=1.2,
               alpha=0.85, label=f"layer {layer}" + (" (backward)" if "reverse" in name else ""))
    ax.set(title=title, xlabel="step", ylabel="‖∇ weight_hh‖₂ (pre-clip)", yscale="log")
    ax.legend(frameon=False, fontsize=7)


def deepest_uni_run(rnn_type: str) -> str | None:
    sub = results[(results.rnn_type == rnn_type) & (results.direction == "uni")]
    return sub.num_layers.idxmax() if len(sub) else None


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, rnn_type in zip(axes, ("gru", "lstm")):
    run = deepest_uni_run(rnn_type)
    if run is None:
        ax.set_title(f"no unidirectional {rnn_type.upper()} run in this grid")
        continue
    plot_gradient_flow(run, ax, f"{int(results.loc[run, 'num_layers'])}-layer unidirectional "
                              f"{rnn_type.upper()} ({run})")
plt.tight_layout()
plt.show()

**→ what to look for (fill in after running):** if layer 0's curve sits visibly below layer 2's throughout training, that is vanishing gradients through the stack; if either curve grows without bound or spikes near the clipping threshold of 1.0, that is exploding gradients (and clipping is doing real work, unlike in point 2's healthy 1-layer baseline). Compare the two panels: GRU's reset/update gates and LSTM's forget gate are both designed to mitigate this, so a visibly flatter, more stable LSTM curve here would be the gradient-flow evidence for preferring LSTM at depth, independent of the CV score in section 3.

_Replace this paragraph with what the plots actually show once the grid has run — which layer is weakest, whether GRU and LSTM differ, and whether that lines up with which one scored better in section 3._

## 7. Error analysis
Every CV text where the best architecture's prediction differs from gold, after `hard_processing` — same format as points 2 and 5, to make the error patterns directly comparable.

In [ ]:
BEST_RUN = results.comp_f1.idxmax()
print("best architecture by CV F1:", BEST_RUN, f"(cv_f1={results.loc[BEST_RUN, 'comp_f1']:.3f})")

best_pred = decode(corpora["cv"], load_probs(BEST_RUN, "cv"))
rows = []
for text, gold, pred in zip(corpora["cv"].texts, corpora["cv"].gold, best_pred):
    g, p = {hard_processing(x) for x in gold}, {hard_processing(x) for x in pred}
    if g != p:
        rows.append({"text": text[:100].replace("\n", " ⏎ "), "missed": sorted(g - p), "extra": sorted(p - g),
                     "correct": sorted(g & p)})
print(f"{len(rows)} of {len(best_pred)} CV texts have an error")
pd.DataFrame(rows)

## 8. Conclusions
_Fill in after running the full grid on Colab — do not report numbers that have not actually been produced by the runs above._

- **Best architecture:** `___` scored CV ___ (es ___), vs the point-5 reference BiGRU1 at CV 0.597 and the point-2 vanilla-RNN baseline at CV 0.488.
- **GRU vs LSTM:** at matched depth and direction, which cell type wins on CV? On es? Does LSTM's larger parameter count (section 5) buy a real CV gain, or only a train-domain one?
- **Depth:** does CV F1 improve, plateau, or get worse from 1 → 2 → 3 layers? If it gets worse, is that visible in the gradient-flow plots (section 6) or is it overfitting (es keeps rising, CV falls — the same pattern as point 2's "more training hurts the test domain" finding, now for "more capacity")?
- **Bidirectional:** does the es-vs-CV gap from point 5 (bidirectional helps es more than CV) hold at every depth, or does it change with more layers?
- **Gradient flow:** summarize what section 6 actually showed — which layer's gradients were weakest, whether GRU or LSTM looked more stable, and whether training ever needed the gradient clip.
- **Failed runs, if any:** note any configuration that did not train cleanly (loss diverged/NaN, `es_f1` collapsed) and the likely cause — per the assignment notes, a documented failure is worth reporting, not hiding.
- **Recommendation for point 4:** the architecture carried forward into the optimizer/scheduler/hyperparameter notebook (loaded automatically there from `results_03_architecture.csv`).

## 9. Results log and submission
All 12 runs go into the shared results log with their CV scores. Review section 3 first, then set `SUBMIT_RUN` to the run you want to spend one of the team's daily Kaggle submissions on (or leave it `None` to only update the log).

In [ ]:
configs = {make_config(*combo)["run_name"]: make_config(*combo) for combo in GRID}
for run_name, report in results.iterrows():
    log_result(configs[run_name], report, notes="architecture")

SUBMIT_RUN = None  # e.g. BEST_RUN, after reviewing section 3 and section 8
lb_scores = read_log().set_index("run_name").lb_public
if QUICK_RUN:
    print("quick run: not submitting")
elif SUBMIT_RUN and pd.isna(lb_scores.get(SUBMIT_RUN)):
    submit(configs[SUBMIT_RUN], results.loc[SUBMIT_RUN], notes="architecture")
read_log().query("point == 3")